# A2.3 · Delegation that narrows, and survives audit

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.2 · Bootstrapping the first credential](https://spbreed.github.io/cyber-commons/lessons/A2.2.html)**.

| | |
|---|---|
| Tools used | SPIFFE/SPIRE, Keycloak, RFC 8693 token exchange, RFC 8705 mTLS binding |

## What this lesson is

**What it covers.** Run both narrowing rules against a request that passes one and fails the other.

**Why a security engineer needs it.** Subset-only lets a privileged user hand an agent authority it must never hold; ceiling-only lets the agent exceed the person who asked. The control it builds is: token exchange that intersects presented scope with the actor's ceiling, and records the chain.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The access token your agent holds is a password: whoever reads it out of a log can spend it. Three published standards fix that — an SVID says which workload is calling, RFC 8693 says on whose authority, and RFC 8705 binds the token to the certificate that earned it. Most estates implement the middle one and stop.

> **At CyberTravels.** Alex can issue refunds; the triage agent must never be able to. Delegation from Alex to CyberTravels has to narrow to a subset of what he presented AND stay inside the receiving agent's own ceiling. R1.

## 2 · The framework

```
   AI agent pod                authorization         downstream
                                  server              service
   +-------------+            +-------------+     +-------------+
   | X.509 SVID  | --mTLS-->  | RFC 8693    | --> | re-derives  |
   | spiffe://.. |  layer 1   | exchange    |     | x5t#S256    |
   |             |            |             |     | from ITS    |
   | user token  | ---------> | sub = user  |     | own TLS     |
   +-------------+  layer 2   | act = agent |     | connection  |
                              | cnf = x5t   |     +-------------+
                              +------+------+        layer 3
                                     |
                     scope issued = requested
                                  & presented       (subset)
                                  & actor ceiling   (never-exceed)
```

**Mitigates: T3 Privilege Compromise · T8 Repudiation · T14 Human Attacks.**

The agent has its own identity now. This lesson is about carrying the user's
authority alongside it — without losing it, amplifying it, or handing it to
whoever picks the token out of a log.

Three layers do that, and each one answers a question the layer above it
cannot. Skipping any of them leaves a specific, named hole.

**Layer 1 — mTLS with an X.509 SVID. *Is this the workload it claims to be?***
The agent pod presents a client certificate whose URI SAN is a SPIFFE ID:
`spiffe://cybertravels.com/ns/prod/sa/agent-alpha`. The platform issues it on
attestation (A2.2), it lives minutes rather than months, and it rotates without
anybody being told. Nothing downstream trusts a name in a header again.

**Layer 2 — OAuth on-behalf-of, RFC 8693 token exchange. *On whose authority?***
The agent presents two tokens: the user's (`subject_token`) and its own SVID
(`actor_token`). The authorization server returns one access token whose `sub`
is still the user and whose **`act` claim** names the agent. Delegation is
written down rather than inferred, and it nests — `act.act` records the hop
before — so `alice → orchestrator → agent-alpha` survives into the audit log.

**Layer 3 — RFC 8705 certificate-bound tokens. *Is the presenter the one it was
issued to?*** The issued token carries a `cnf` claim holding `x5t#S256`: the
SHA-256 thumbprint of the client certificate that asked for it. The downstream
service recomputes the thumbprint of the certificate on *its own* TLS
connection and compares. A bearer token is a password; a bound token is useless
to anyone holding it without the private key that earned it.

On top of the three layers, delegation still has to **narrow**, and it has to
satisfy two rules rather than one:

**Subset of presented.** The issued token carries no more scope than the
incoming one. Stops the agent inventing authority.

**Within the actor's ceiling.** The issued token carries no more than the
receiving agent may ever hold. Stops a privileged user handing an agent
authority the agent must never have.

Neither is sufficient alone, and they fail in opposite directions. Subset-only
lets an admin's request give a low-trust agent `payments:refund` — legitimately,
and it looks correct in every log. Ceiling-only lets an agent exceed the person
who asked, which is A1.6. **The issued scope is the intersection.**

> **On the standards.** RFC 8693 (token exchange) and RFC 8705 (mTLS client
> authentication and certificate-bound access tokens) are published and widely
> implemented — none of this is future work. The IETF OAuth working group
> additionally has a live draft for AI agents acting on a user's behalf, which
> adds agent-specific metadata to the same exchange. It is a draft, not an RFC,
> and nothing in this lesson depends on it: all three layers are buildable on
> RFC 8693 and RFC 8705 as they stand today.

<div style="display:flex;gap:6px;align-items:stretch;flex-wrap:wrap;font-family:ui-sans-serif,system-ui,-apple-system,Segoe UI,Roboto,sans-serif;margin:6px 0 2px"><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">agent pod</div><div style="border:1px solid rgba(77,155,255,.55);border-left:3px solid #4D9BFF;border-radius:8px;background:rgba(77,155,255,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#129302;</span> AI agent</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">holds an X.509 SVID issued on attestation, plus the user&#x27;s token from the request</div><div style="font-size:10px;color:#4D9BFF;margin-top:5px;font-weight:600;letter-spacing:.02em">spiffe://.../sa/agent-alpha</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">layer 1 · mTLS</div><div style="border:1px solid rgba(63,160,107,.55);border-left:3px solid #3FA06B;border-radius:8px;background:rgba(63,160,107,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128274;</span> client certificate</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">URI SAN carries the SPIFFE ID. Minutes long, rotated automatically</div><div style="font-size:10px;color:#3FA06B;margin-top:5px;font-weight:600;letter-spacing:.02em">REFUSES A NAME IN A HEADER</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">layer 2 · RFC 8693</div><div style="border:1px solid rgba(224,145,47,.55);border-left:3px solid #E0912F;border-radius:8px;background:rgba(224,145,47,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#127915;&#65039;</span> authorization server</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">subject_token is the user, actor_token is the SVID. One token comes back: sub is still the user, act names the agent</div><div style="font-size:10px;color:#E0912F;margin-top:5px;font-weight:600;letter-spacing:.02em">REFUSES WIDENING</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">layer 3 · RFC 8705</div><div style="border:1px solid rgba(224,145,47,.55);border-left:3px solid #E0912F;border-radius:8px;background:rgba(224,145,47,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128273;</span> cnf / x5t#S256</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">the thumbprint of the certificate that asked for the token, stamped into the token itself</div><div style="font-size:10px;color:#E0912F;margin-top:5px;font-weight:600;letter-spacing:.02em">REFUSES A STOLEN TOKEN</div></div></div><div style="align-self:center;color:rgba(138,147,166,.8);font-size:20px;padding:0 2px;flex:0 0 auto">&#8250;</div><div style="flex:1 1 150px;min-width:150px"><div style="font-size:10px;text-transform:uppercase;letter-spacing:.09em;color:#8A93A6;font-weight:600;padding-bottom:5px;border-bottom:1px solid rgba(138,147,166,.4);margin-bottom:3px">downstream</div><div style="border:1px solid rgba(224,92,75,.55);border-left:3px solid #E05C4B;border-radius:8px;background:rgba(224,92,75,.10);padding:8px 11px;margin:5px 0;min-width:132px"><div style="font-size:13px;font-weight:600"><span style="font-size:15px">&#128179;</span> payments API</div><div style="font-size:11px;color:#8A93A6;margin-top:2px;line-height:1.35">re-derives the thumbprint from its own TLS connection and compares it before reading a single scope</div><div style="font-size:10px;color:#E05C4B;margin-top:5px;font-weight:600;letter-spacing:.02em">R1</div></div></div></div><div style="font-size:12px;color:#8A93A6;margin-top:8px;line-height:1.5">Each layer answers a question the one before it cannot: is this the workload, on whose authority, and is the presenter the one the token was issued to. The code below builds all three, then steals the token.</div>

> **What this control closes.**
>
> Both narrowing rules, at every hop, plus a binding that makes the token worthless off the connection that earned it.

## 3 · The same three layers, against real Keycloak

Everything above is the protocol, modelled in the standard library so this
notebook runs with the internet switched off. That proves RFC 8693 and RFC 8705
work. It does not prove the product you are about to deploy implements them,
which is a different question and has a different answer.

[`labs/tools/keycloak-obo/`](https://github.com/spbreed/cyber-commons/tree/main/labs/tools/keycloak-obo)
downloads Keycloak 26.0.7, starts it with mTLS, configures this realm and runs
the same three checks. Three of them behaved as the specifications describe:

```
the agent asks for its own token over plain HTTP, with no certificate:
  {"error":"invalid_request",
   "error_description":"Client Certification missing for MTLS HoK Token Binding"}

its x5t#S256 thumbprint (computed with openssl):
  BuTPvYMaI3z-suLCcWsnFHDCv_6VQdDrYwLlf70Sjfg
the cnf claim on the token Keycloak issued:
  {"x5t#S256": "BuTPvYMaI3z-suLCcWsnFHDCv_6VQdDrYwLlf70Sjfg"}

the same token, presented to a resource server that compares:
  legitimate agent  HTTP 200
  the thief         HTTP 403  cnf mismatch - token was not issued to this client
  no client cert    HTTP 403  no client certificate presented
```

The thief's certificate is signed by the same CA and is therefore trusted.
Trust is not what separates them.

**And two did not.** These are the ones worth carrying out of this lesson:

**Keycloak's standard token exchange emits no `act` claim** — with or without
an `actor_token`. It returns 200 either way and produces the same token. `azp`
does name the agent, so the information is not lost, but `azp` is one value and
`act` nests: a three-hop chain has nowhere to go. The delegation chain
reconstructed in the next cell is something you write a protocol mapper for.

**The exchange drops the certificate binding.** The direct token carries `cnf`.
The exchanged token — requested over the same mTLS connection, with the same
certificate — does not. So the token the agent actually carries downstream, the
one issued for acting on alice's behalf, is a bearer token again, and the theft
the binding was bought to prevent is back.

Neither is a reason not to build this. Both are reasons to check your own
deployment rather than assume the control is on because the feature exists.

## 4 · What the audit trail can now answer

Every hop is on the token, so the chain reconstructs from the token alone rather than by correlating four services' logs on timestamp. This is the thing A1.14 could not do.

## 5 · Verifying the chain, as a skill

The two findings above are what you get from *checking* a deployment rather than reading its design document, and CyberTravels has four agents and a payments API to check. The procedure walks every hop — user, agent, MCP server, tool, downstream — looks for token passthrough, and refuses to accept a matching `sub` as proof of delegation, because impersonation produces one too. This is the file in this repository:

In [ ]:
# skills/attestation/identity-chain-verifier/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: identity-chain-verifier
description: >-
  Verify workload identity and end-to-end on-behalf-of propagation across
  user, agent, MCP server, tool and downstream. Use to check for token
  passthrough, to distinguish delegation from impersonation, or when asked
  whether a downstream can attribute an action to the original user.
allowed-tools: Bash, Read, Grep
---

# Identity Chain Verifier

**Controls:** Control 3 — workload identity and OBO attribution

## Confidence: HIGH

Delegation is cryptographically distinguishable from impersonation, and token
passthrough is visible in the audience claim. This control is genuinely
verifiable.

## Procedure

1. **Establish workload identity.** Read the registration entry and its
   selectors. For an X.509 credential the identity is in the SAN URI; for a JWT
   credential it is the subject. Confirm it matches the deployment manifest.

2. **Walk every hop.** For user → agent → MCP server → tool → downstream,
   record the principal, the audience, and the token type at each hop.

3. **Check for token passthrough — the finding that matters most.** A server
   must not forward the token it received to a downstream. Each hop must carry
   a **distinct, audience-scoped** token. A repeated audience across two hops
   is passthrough, and it is the mechanism behind the confused-deputy class.

4. **Distinguish delegation from impersonation.** In token exchange, delegation
   is impossible without an actor token: the presence of an actor token, and a
   corresponding actor claim in the issued token, is what proves the chain is
   delegation rather than the agent simply becoming the user.

5. **Check credential hygiene.** Short lifetime, rotation, and — where mutual
   TLS is possible — prefer certificate credentials over bearer tokens, which
   are replayable. Flag every JWT-only hop.

## Output contract

```json
{
  "deployment_id": "str",
  "chain": [
    {"hop": "user|agent|mcp|tool|downstream", "principal": "str",
     "audience": "str", "token_type": "x509|jwt",
     "mode": "delegation|impersonation|passthrough", "ttl_seconds": 0}
  ],
  "passthrough_violations": [{"from_hop": "str", "to_hop": "str", "audience": "str"}],
  "delegation_proven": true,
  "jwt_only_hops": ["str"],
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Accepting a matching subject as proof of delegation.** Impersonation also
  produces the right subject. The actor claim is the difference.
- **Not checking the audience.** Passthrough is invisible without it.
- **Treating a long-lived JWT as equivalent to a rotated certificate.**
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/attestation/identity-chain-verifier/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/attestation/identity-chain-verifier/scripts/identity_chain_verifier.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Exchange a token so the subject stays the user and the actor names the agent, then check the certificate binding downstream.

This is the executable half of the `identity-chain-verifier` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import base64, hashlib, hmac, json

def b64u(raw: bytes) -> str:
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode()

# --- layer 1: the SVID -----------------------------------------------------
# A real X.509-SVID is a certificate carrying the SPIFFE ID in its URI SAN.
# Here it is the DER bytes standing in for one. The only property the protocol
# needs is that the thumbprint is DERIVED from the certificate rather than
# asserted alongside it.
class SVID:
    def __init__(self, spiffe_id, der):
        self.spiffe_id, self.der = spiffe_id, der
    @property
    def thumbprint(self):                        # RFC 8705 x5t#S256
        return b64u(hashlib.sha256(self.der).digest())

AGENT = SVID("spiffe://cybertravels.com/ns/prod/sa/agent-alpha",
             b"cert-agent-alpha")

# --- layer 2: RFC 8693 token exchange --------------------------------------
CEILINGS = {                        # what each actor may EVER hold
 "alice@cybertravels.com":
    {"bookings:read", "bookings:write", "payments:refund"},
 "spiffe://cybertravels.com/ns/prod/sa/orchestrator":
    {"bookings:read", "bookings:write"},
 "spiffe://cybertravels.com/ns/prod/sa/agent-alpha":
    {"bookings:read"},
}
SIGNING_KEY = b"demo-key-not-a-secret"

class DelegationError(Exception): pass

def exchange(subject_token, actor, requested):
    """RFC 8693: subject_token is the user, actor_token is the agent's SVID."""
    requested = set(requested)
    presented = set(subject_token["scope"].split())
    if not requested <= presented:                                 # rule 1
        raise DelegationError(
            f"widening: {sorted(requested - presented)} was never presented")
    ceiling = CEILINGS[actor.spiffe_id]
    issued = requested & ceiling                                   # rule 2
    if issued != requested:
        print(f"   ceiling narrowed it: {actor.spiffe_id.rsplit('/', 1)[-1]} "
              f"may never hold {sorted(requested - ceiling)}")
    act = {"sub": actor.spiffe_id}
    if "act" in subject_token:                    # nest the previous hop
        act["act"] = subject_token["act"]
    return {
      "sub": subject_token["sub"],                # STILL the human
      "aud": "https://payments.cybertravels.internal",
      "scope": " ".join(sorted(issued)),
      "act": act,                                 # the agent, and the chain
      "cnf": {"x5t#S256": actor.thumbprint},      # layer 3, stamped here
    }

def sign(claims):
    head = b64u(json.dumps({"alg": "HS256", "typ": "JWT"}, sort_keys=True).encode())
    body = b64u(json.dumps(claims, sort_keys=True).encode())
    mac = hmac.new(SIGNING_KEY, f"{head}.{body}".encode(), hashlib.sha256)
    return f"{head}.{body}.{b64u(mac.digest())}"

user = {"sub": "alice@cybertravels.com",
        "scope": "bookings:read bookings:write payments:refund"}
tok = exchange(user, AGENT, {"bookings:read"})

print("the access token the payments API will actually see:\n")
print(json.dumps(tok, indent=2, sort_keys=True))
print(f"\nas a JWT: {sign(tok)[:78]}...")

# The token above leaked. A debug log, a crash dump, an LLM transcript - it
# does not matter which. Another pod picks it up and replays it.
THIEF = SVID("spiffe://cybertravels.com/ns/prod/sa/scraper", b"cert-scraper")

def serve_bearer(token, _tls_peer):
    """How most services check a token today: is it signed, does it say yes?"""
    return "bookings:read" in token["scope"].split()

def serve_bound(token, tls_peer):
    """RFC 8705: re-derive the thumbprint from THIS connection, first."""
    want = token.get("cnf", {}).get("x5t#S256")
    if want is None:
        raise PermissionError("token is not certificate-bound - refusing")
    if not hmac.compare_digest(want, tls_peer.thumbprint):
        raise PermissionError(
            f"cnf mismatch: issued to {want[:12]}..., presented on a "
            f"connection using {tls_peer.thumbprint[:12]}...")
    return "bookings:read" in token["scope"].split()

print("the legitimate agent, on its own connection:")
print(f"   bearer check : {serve_bearer(tok, AGENT)}")
print(f"   bound  check : {serve_bound(tok, AGENT)}")

print("\nthe same token, replayed by a different pod:")
print(f"   bearer check : {serve_bearer(tok, THIEF)}   <- accepted. a bearer "
      f"token is a password.")
try:
    serve_bound(tok, THIEF)
except PermissionError as e:
    print(f"   bound  check : refused - {e}")

# And the widening attempt: alice really does hold payments:refund, but
# agent-alpha may never hold it, no matter who asks.
print("\nalice asks agent-alpha to issue a refund on her behalf:")
subset_ok = {"payments:refund"} <= set(user["scope"].split())
refund = exchange(user, AGENT, {"payments:refund"})
print(f"   subset-of-presented alone would allow it : {subset_ok}")
print(f"   scope actually issued                    : {refund['scope'] or 'none'}")

assert serve_bearer(tok, THIEF) is True          # the hole
try:
    serve_bound(tok, THIEF)
    raise AssertionError("the binding did not hold")
except PermissionError:
    pass
assert subset_ok and refund["scope"] == ""
assert tok["act"]["sub"].endswith("/sa/agent-alpha")

ORCH = SVID("spiffe://cybertravels.com/ns/prod/sa/orchestrator",
            b"cert-orchestrator")

hop1 = exchange(user, ORCH, {"bookings:read", "bookings:write"})
hop2 = exchange(hop1, AGENT, {"bookings:read"})

def chain(claims):
    """sub is the human; act nests one entry per hop, most recent first."""
    hops, node = [], claims.get("act")
    while node:
        hops.append(node["sub"].rsplit("/", 1)[-1])
        node = node.get("act")
    return " -> ".join([claims["sub"], *reversed(hops)])

print(f"delegation chain : {chain(hop2)}")
print(f"final scope      : {hop2['scope']}")
print(f"bound to         : {hop2['cnf']['x5t#S256'][:16]}...  "
      f"(agent-alpha's certificate, not the orchestrator's)")
assert chain(hop2) == ("alice@cybertravels.com -> orchestrator -> agent-alpha")
assert hop2["cnf"]["x5t#S256"] == AGENT.thumbprint

## What you just proved

The verifier skill loads and reports its shape: the description an agent routes on, the tools it may use, and a procedure that walks every hop of the chain rather than checking the token it was handed. Read its failure modes against the Keycloak findings above — a matching `sub` is not delegation, and a chain with no audience check cannot see passthrough at all.

## Your turn

Find your token exchange and check three things: does it set an `act` claim, does it check the actor's ceiling as well as the subset rule, and does anything downstream look at `cnf`? Most implementations do the subset rule only — it is the one the specification example shows.

---

**Next → [A2.4 · Just-in-time authority](https://spbreed.github.io/cyber-commons/lessons/A2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*